### 1. Load high confidence places from `holinshed_places.csv`.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sys

sys.path.append("../name_extraction")
from helpers import load_places, load_texts, count_names

# Very high confidence places (confidence >= 0.75) from the source of truth, highest confidence first
places = load_places("../name_extraction/entity_names/holinshed_places.csv")
print(f"Working with {len(places)} high confidence places.")

### 2. Read each reign from `monarchs_reign.xml`, then use the shared functions in `name_extraction/helpers.py` to normalize early-modern spellings in both the text and the place names, clean up the place labels, merge spellings of the same place, and count every place's mentions in every reign to build the place-by-reign count matrix.

In [ ]:
# Text of each regnal entry, in XML order
texts = load_texts("../1577_files/monarchs_reign.xml")
if not texts:
    raise ValueError("No regnal entries found in monarchs_reign.xml")

# Word counts per reign, used later to normalize mentions
word_counts = pd.Series(
    {reign: len(text.split()) for reign, text in texts.items()}, name="word_count"
)

print(
    f"Searching {len(places)} places across {len(texts)} regnal entries (~90 seconds)",
    flush=True,
)

# count_names does all the normalizing (aliases, labels, etc.)
place_counts_by_reign = count_names(
    places,
    texts,
).rename_axis(index="place_name", columns="file")

print("Finished processing.", flush=True)

place_counts_by_reign.info()

### 3. Chart the five most-mentioned places overall, report per-reign coverage, and plot a heatmap.

In [ ]:
# Top 5 most-mentioned places (bar chart)
place_totals = place_counts_by_reign.sum(axis=1).sort_values(ascending=False)
top_5_places = place_totals.head(5)

plt.figure(figsize=(5, 3))
ax = sns.barplot(
    x=top_5_places.values,
    y=top_5_places.index,
    hue=top_5_places.index,
    legend=False,
)

plt.title("Top 5 Most-Mentioned Places (All Reigns)")
plt.xlabel("Total mentions")
plt.ylabel("Place")
plt.tight_layout()
plt.show()

# Number of distinct places with at least one match in each reign
places_with_matches = (place_counts_by_reign > 0).sum()

# Total number of place mentions in each reign
total_mentions = place_counts_by_reign.sum()

# Word counts per reign, ordered to match the count matrix
word_counts = word_counts.reindex(place_counts_by_reign.columns)

coverage = pd.DataFrame({
    "word_count": word_counts,
    "places_found": places_with_matches,
    "total_mentions": total_mentions,
})

print("Coverage:")
display(coverage)

# Top 20 places by total mentions
top_20_places = place_totals.head(10).index
top_20_by_reign = place_counts_by_reign.loc[top_20_places]

# Normalize by the word count of each reign
mentions_per_word = top_20_by_reign.div(word_counts, axis=1)

# Heatmap of normalized place mentions
plt.figure(figsize=(16, 8))

ax = sns.heatmap(
    mentions_per_word,
    cmap="Greens",
    linewidths=0.5,
)

ax.set_title("Top 10 Places Across Reigns (Mentions per Word)")
ax.set_xlabel("Reign")
ax.set_ylabel("Place")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

### 4. List the fifteen darkest heatmap cells.

In [ ]:
hot_spots = (
    mentions_per_word.stack()
    .rename("mentions_per_word")
    .rename_axis(["place", "reign"])
    .reset_index()
)

hot_spots["mentions"] = [
    top_20_by_reign.loc[place, reign]
    for place, reign in zip(hot_spots["place"], hot_spots["reign"])
]
hot_spots["per_1000_words"] = (hot_spots["mentions_per_word"] * 1000).round(2)

top_hot_spots = (
    hot_spots.sort_values("mentions_per_word", ascending=False)
    .head(15)[["place", "reign", "mentions", "per_1000_words"]]
    .reset_index(drop=True)
)
top_hot_spots.index += 1

print(top_hot_spots.to_string())

### 5. Give the single most-mentioned place in each reign, with its rate per 1,000 words.

In [ ]:
# Most-mentioned place in each reign (one row per reign)
hot_spot_per_reign = pd.DataFrame({
    "place": place_counts_by_reign.idxmax(axis=0),
    "mentions": place_counts_by_reign.max(axis=0),
}).rename_axis("reign").reset_index()

hot_spot_per_reign["per_1000_words"] = (
    hot_spot_per_reign["mentions"]
    .div(word_counts.reindex(hot_spot_per_reign["reign"]).to_numpy())
    .mul(1000)
    .round(2)
)
hot_spot_per_reign.index += 1

print(hot_spot_per_reign.to_string())